# Data Cleaning - Interruption

The data will come from the text column of `interruption_bronze`. The outage data will be filtered by only selecting posts containing "BENECO POWER SERVICE ADVISORY".

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.silver.interruption_silver
AS SELECT
    text
FROM beneco_pipeline.bronze.interruption_bronze
WHERE text LIKE "%BENECO POWER SERVICE ADVISORY%" AND text RLIKE '(?i)POWER INTERRUPTION';

### Clean text column
Power service advisories are manually created, which often leads to inconsistencies such as formatting issues, typographical errors, and irregular spacing.

To standardize and normalize the extracted data, the following transformations are applied:

* **Correct spelling and grammar**  
  Apply the `ai_fix_grammar` function to automatically correct spelling and grammatical errors in the text.
* **Remove extra spaces**  
  Eliminate double spaces to ensure consistent text formatting.
* **Standardize time format**  
  Insert a space between time values and the AM/PM indicator (e.g., convert `1:00PM → 1:00 PM`).
* **Unicode conversion**  
  Convert unicode formatting to normal text using `unicodedata.normalize` function.


In [0]:
%sql
-- AI Grammar Correction
CREATE OR REPLACE TABLE beneco_pipeline.silver.interruption_silver
AS SELECT
    ai_fix_grammar(text) AS text
FROM beneco_pipeline.silver.interruption_silver;

-- Double space removal
UPDATE beneco_pipeline.silver.interruption_silver
SET text = regexp_replace(text,'  ',' ')
WHERE text like '%  %';

-- Add space between h:mm & AM/PM
UPDATE beneco_pipeline.silver.interruption_silver
SET text = regexp_replace(text, r'(?i)(\d{1,2}:\d{2})(AM|PM)', 
regexp_extract(text, r'(?i)(\d{1,2}:\d{2})(?:AM|PM)', 1)||' '||regexp_extract(text, r'(?i)\d{1,2}:\d{2}(AM|PM)',1));

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import unicodedata

# define normalization fucntion and make into udf
def normalize_text(text):
    normalized = unicodedata.normalize('NFKD', text)
    return normalized
normalize_udf = udf(normalize_text, StringType())

# read table and normalize text column using udf
df = spark.read.table("beneco_pipeline.silver.interruption_silver")
df = df.withColumn("text", normalize_udf("text"))

# write into table
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("beneco_pipeline.silver.interruption_silver")

In [0]:
%sql
SELECT * FROM beneco_pipeline.silver.interruption_silver
WHERE TEXT LIKE '%48PM%';